In [1]:
# ==== ProtVar paging + flattening: setup (refactored) ====
from __future__ import annotations
import os, json, time, datetime as dt
from typing import Any, Dict, List, Optional, Iterable, Tuple

import requests
import pandas as pd

PROTVAR_MAP_BY_ACC = "https://www.ebi.ac.uk/ProtVar/api/mapping/accession/{acc}?page={page}&pageSize={page_size}"

# -------- IO helpers --------
def _save_json(obj: Any, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def _load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# -------- HTTP client --------
class ProtVarClient:
    """
    Minimal client to page through ProtVar's mapping-by-accession endpoint.
    Caches all pages to a single JSON file for reuse.
    """
    def __init__(self, base_url_template: str = PROTVAR_MAP_BY_ACC, timeout: int = 30):
        self.base = base_url_template
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({"Accept": "application/json"})

    def fetch_page(self, acc: str, page: int, page_size: int = 1000) -> Dict[str, Any]:
        url = self.base.format(acc=acc, page=page, page_size=page_size)
        r = self.session.get(url, timeout=self.timeout)
        r.raise_for_status()
        return r.json()

    def fetch_all(
        self,
        acc: str,
        page_size: int = 1000,
        cache_path: Optional[str] = None,
        force: bool = False,
        pause: float = 0.05,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        if cache_path is None:
            cache_path = f"{acc}_protvar.json"

        if os.path.exists(cache_path) and not force:
            if verbose:
                print(f"Using cached ProtVar data: {cache_path}")
            return _load_json(cache_path)

        pages: List[Dict[str, Any]] = []
        page = 1
        total_variants: Optional[int] = None

        if verbose:
            print(f"Fetching ProtVar variants for {acc} with page size {page_size}...")

        while True:
            data = self.fetch_page(acc, page=page, page_size=page_size)
            pages.append(data)

            if total_variants is None:
                for key in ("totalVariants", "total", "totalItems", "totalCount"):
                    if isinstance(data, dict) and key in data and isinstance(data[key], int):
                        total_variants = data[key]
                        break

            n_items = _estimate_items_in_page(data)
            if verbose:
                tv = total_variants if total_variants is not None else "?"
                print(f"  page {page}: {n_items} items (total {tv})")

            # Stop at last page or once we have at least the reported total.
            if n_items == 0 or n_items < page_size:
                break
            if total_variants is not None:
                fetched_so_far = sum(_estimate_items_in_page(p) for p in pages)
                if fetched_so_far >= total_variants:
                    break

            page += 1
            if pause:
                time.sleep(pause)

        bundle = {
            "accession": acc,
            "fetched_at": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "page_size": page_size,
            "pages": pages,
        }
        _save_json(bundle, cache_path)
        if verbose:
            print(f"Saved {len(pages)} page(s) → {cache_path}")
        return bundle

# -------- Schema-tolerant finders --------
def _estimate_items_in_page(page_json: Dict[str, Any]) -> int:
    for key in ("results", "items", "content", "data"):
        if key in page_json:
            val = page_json[key]
            if isinstance(val, dict) and "inputs" in val and isinstance(val["inputs"], list):
                return len(val["inputs"])
            if isinstance(val, list):
                return len(val)
    if "inputs" in page_json and isinstance(page_json["inputs"], list):
        return len(page_json["inputs"])
    return 0

def _iter_all_inputs_from_pages(pages: List[Dict[str, Any]]) -> Iterable[Dict[str, Any]]:
    for page in pages:
        yield from _find_inputs_anywhere(page)

def _find_inputs_anywhere(obj: Any) -> Iterable[Dict[str, Any]]:
    if isinstance(obj, dict):
        if "inputs" in obj and isinstance(obj["inputs"], list):
            for item in obj["inputs"]:
                if isinstance(item, dict):
                    yield item
        for v in obj.values():
            yield from _find_inputs_anywhere(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from _find_inputs_anywhere(v)

def _strip_version(ensembl_id: Optional[str]) -> Optional[str]:
    if not ensembl_id or not isinstance(ensembl_id, str):
        return ensembl_id
    return ensembl_id.split(".")[0]

def _safe_get(d: Dict[str, Any], path: List[str]) -> Any:
    cur = d
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return None
        cur = cur[p]
    return cur

def _collect_protein_tx_refs(isoform: Dict[str, Any]) -> Tuple[List[str], List[str]]:
    ensp_list, enst_list = [], []
    for ts in isoform.get("translatedSequences", []) or []:
        if "ensp" in ts and ts["ensp"]:
            ensp_list.append(ts["ensp"])
        for t in ts.get("transcripts", []) or []:
            enst = t.get("enst")
            if enst:
                enst_list.append(enst)
    return list(dict.fromkeys(ensp_list)), list(dict.fromkeys(enst_list))

# -------- New: protein summary extractor --------
def protvar_protein_summary(
    bundle: Dict[str, Any],
    accession: Optional[str] = None,
    canonical_only: bool = True,
) -> Dict[str, Optional[str]]:
    """
    Extract protein-level facts once, so we do not repeat them in the table.
    Returns a dict with keys:
      Protein accession, Protein name, Gene, Ensembl gene, Chromosome
    If multiple values are present across items, they are joined with '; '.
    """
    accs, names, genes, ensgs, chrs = set(), set(), set(), set(), set()

    for inp in _iter_all_inputs_from_pages(bundle.get("pages", [])):
        chrom = inp.get("chr")
        if chrom:
            chrs.add(str(chrom))
        for m in inp.get("mappings", []) or []:
            for g in m.get("genes", []) or []:
                if g.get("geneName"):
                    genes.add(g.get("geneName"))
                if g.get("ensg"):
                    ensgs.add(_strip_version(g.get("ensg")))
                for iso in g.get("isoforms", []) or []:
                    if accession and iso.get("accession") != accession:
                        if canonical_only and not iso.get("canonical", False):
                            continue
                        # If we are strict on accession, skip others
                        continue
                    if iso.get("accession"):
                        accs.add(iso.get("accession"))
                    if iso.get("proteinName"):
                        names.add(iso.get("proteinName"))

    # Prefer the base accession (without isoform) if present
    protein_accession = accession or (sorted(accs)[0] if accs else None)

    return {
        "Protein accession": protein_accession,
        "Protein name": "; ".join(sorted(names)) if names else None,
        "Gene": "; ".join(sorted(genes)) if genes else None,
        "Ensembl gene": "; ".join(sorted(ensgs)) if ensgs else None,
        "Chromosome": "; ".join(sorted(chrs)) if chrs else None,
    }

def print_protein_summary(summary: Dict[str, Optional[str]]) -> None:
    # Pretty one-liner and a compact block for clarity
    one_line = " | ".join(f"{k}: {v}" for k, v in summary.items() if v)
    if one_line:
        print(one_line)
    else:
        print("No protein summary available.")

def protvar_to_dataframe(
    bundle: Dict[str, Any],
    accession: Optional[str] = None,
    canonical_only: bool = True,
) -> pd.DataFrame:
    """
    Flatten the cached pages into a tidy DataFrame with VARIANT-LEVEL columns only.
    Protein-level facts are NOT repeated here (they are printed once via protvar_protein_summary).

    Kept columns (human readable):
      Position, Reference AA, Variant AA, Consequence,
      Codon change, Amino acid change,
      Genomic position, Reference allele, Alternate allele,
      CADD, Allele frequency, Conservation score, AM pathogenicity, AM class, ESM score,
      ENSP, ENST

    Notes:
      - ENSP/ENST are preserved for cross-referencing.
      - 'Canonical accession', 'Function URI', 'Population URI' are removed.
      - 'Accession', 'Protein name', 'Gene', 'Ensembl gene', 'Chromosome' are removed here and printed once instead.
    """
    pages = bundle.get("pages", [])
    rows: List[Dict[str, Any]] = []

    for inp in _iter_all_inputs_from_pages(pages):
        chrom = inp.get("chr")
        gpos = inp.get("pos")
        mappings = inp.get("mappings", [])
        for m in mappings:
            genes = m.get("genes", [])
            for g in genes:
                ref_nt = g.get("refAllele")
                alt_nt = g.get("altAllele")
                cadd = g.get("caddScore")
                af = g.get("alleleFreq")

                for iso in g.get("isoforms", []) or []:
                    acc = iso.get("accession")
                    if accession and acc != accession:
                        continue
                    if canonical_only and not iso.get("canonical", False):
                        continue

                    conserv = _safe_get(iso, ["conservScore", "score"])
                    am_path = _safe_get(iso, ["amScore", "amPathogenicity"])
                    am_class = _safe_get(iso, ["amScore", "amClass"])
                    esm = _safe_get(iso, ["esmScore", "score"])

                    ensp_list, enst_list = _collect_protein_tx_refs(iso)

                    rows.append({
                        "Position": iso.get("isoformPosition"),
                        "Reference AA": iso.get("refAA"),
                        "Variant AA": iso.get("variantAA"),
                        "Consequence": iso.get("consequences"),
                        "Codon change": iso.get("codonChange"),
                        "Amino acid change": iso.get("aminoAcidChange"),

                        "Genomic position": gpos,
                        "Reference allele": ref_nt,
                        "Alternate allele": alt_nt,

                        "CADD": cadd,
                        "Allele frequency": af,
                        "Conservation score": conserv,
                        "AM pathogenicity": am_path,
                        "AM class": am_class,
                        "ESM score": esm,

                        "ENSP": "; ".join(ensp_list) if ensp_list else None,
                        "ENST": "; ".join(enst_list) if enst_list else None,
                    })

    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

    # Order columns for readability
    preferred_order = [
        "Position", "Reference AA", "Variant AA", "Consequence",
        "Codon change", "Amino acid change",
        "Genomic position", "Reference allele", "Alternate allele",
        "CADD", "Allele frequency", "Conservation score",
        "AM pathogenicity", "AM class", "ESM score",
        "ENSP", "ENST",
    ]
    cols = [c for c in preferred_order if c in df.columns] + [c for c in df.columns if c not in preferred_order]
    df = df.reindex(columns=cols)

    # Sort by position if we have it
    if "Position" in df.columns:
        df = df.sort_values(["Position", "Variant AA"], kind="stable")
    return df.reset_index(drop=True)

# ==== AlphaFold DB model URL + downloader ====

AFDB_MODEL_URL_TMPL = "https://alphafold.ebi.ac.uk/files/AF-{acc}-F{fragment}-model_{version}.{ext}"

def afdb_model_url(acc: str, version: str = "v6", fragment: int = 1, ext: str = "cif") -> str:
    """
    Build the AlphaFold DB file URL, e.g.
    AF-O15552-F1-model_v6.cif
    """
    return AFDB_MODEL_URL_TMPL.format(acc=acc, fragment=fragment, version=version, ext=ext)

def download_af_model(
    acc: str,
    dest_dir: str = "models",
    version: str = "v6",
    fragment: int = 1,
    ext: str = "cif",
    force: bool = False,
    timeout: int = 60,
) -> str:
    """
    Download the AlphaFold model file (mmCIF by default) for a UniProt accession.
    Returns the local file path. Uses a simple cache: skips download if the file exists unless force=True.
    """
    import os
    import requests

    url = afdb_model_url(acc, version=version, fragment=fragment, ext=ext)
    os.makedirs(dest_dir, exist_ok=True)
    out_path = os.path.join(dest_dir, f"AF-{acc}-F{fragment}-model_{version}.{ext}")

    if os.path.exists(out_path) and not force:
        print(f"Using cached AlphaFold model: {out_path}")
        return out_path

    print(f"Downloading AlphaFold model\n  URL: {url}\n  → {out_path}")
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        tmp_path = out_path + ".part"
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 15):
                if chunk:
                    f.write(chunk)
        os.replace(tmp_path, out_path)

    size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f"Saved {out_path}  ({size_mb:.2f} MB)")
    return out_path

In [2]:
# ==== ProtVar bulk download (async job + polling) ====
import os, time, json, requests
from typing import Optional

PROTVAR_DOWNLOAD_URL = "https://www.ebi.ac.uk/ProtVar/api/download"

def protvar_bulk_download_for_accession(
    acc: str,
    include_function: bool = True,
    include_population: bool = True,
    include_structure: bool = False,
    assembly: str = "AUTO",
    dest_dir: str = "protvar_downloads",
    filename: Optional[str] = None,
    force: bool = False,
    request_timeout: int = 60,
    poll_timeout: int = 300,   # total seconds to wait for job to be ready
    poll_every: float = 2.0,   # seconds between polls
) -> str:
    """
    POST /download (body = accession as text), then poll the job URL until the file is ready.
    Saves the file and returns its path. Caches by default.
    """
    os.makedirs(dest_dir, exist_ok=True)
    flag_bits = []
    if include_function: flag_bits.append("fun")
    if include_population: flag_bits.append("pop")
    if include_structure: flag_bits.append("str")
    flag_part = "-".join(flag_bits) or "basic"

    if filename is None:
        filename = f"{acc}_{flag_part}.tsv"
    out_path = os.path.join(dest_dir, filename)

    if os.path.exists(out_path) and not force:
        print(f"Using cached ProtVar download: {out_path}")
        return out_path

    # Compose params as the Swagger UI shows
    params = {
        "inputType": "PROTEIN_ACCESSION",
        "function": str(include_function).lower(),
        "population": str(include_population).lower(),
        "structure": str(include_structure).lower(),
        "assembly": assembly,
        # Give each job a unique name to avoid server-side collisions
        "jobName": f"{acc}-{flag_part}-{int(time.time())}",
    }

    headers = {
        "Accept": "*/*",
        "Content-Type": "text/plain",  # body is the accession as plain text
    }

    print("Requesting ProtVar bulk download job…")
    print(f"  POST {PROTVAR_DOWNLOAD_URL}  params={params}  body='{acc}'")
    r = requests.post(
        PROTVAR_DOWNLOAD_URL,
        params=params,
        data=acc.encode("utf-8"),
        headers=headers,
        timeout=request_timeout,
    )
    if r.status_code == 405:
        raise RuntimeError("ProtVar /download must be called with POST (not GET).")
    r.raise_for_status()

    # The POST usually returns JSON describing the job
    job_url = None
    try:
        meta = r.json()
        job_url = meta.get("url")
        if not job_url:
            # fall back to /download/{downloadId}
            dlid = meta.get("downloadId")
            if dlid:
                job_url = f"{PROTVAR_DOWNLOAD_URL}/{dlid}"
    except Exception:
        # Rare case: server streams the file immediately
        cd = r.headers.get("Content-Disposition", "")
        if "filename=" in cd:
            suggested = cd.split("filename=")[-1].strip('"; ')
            out_path = os.path.join(dest_dir, suggested or filename)
        tmp = out_path + ".part"
        with open(tmp, "wb") as f:
            f.write(r.content)
        os.replace(tmp, out_path)
        print(f"Saved ProtVar download → {out_path}")
        return out_path

    if not job_url:
        raise RuntimeError("ProtVar did not return a job URL or file.")

    # Poll until the job URL returns the file (attachment) rather than JSON status
    print(f"Polling job: {job_url}")
    deadline = time.time() + poll_timeout
    last_status = None
    while time.time() < deadline:
        rr = requests.get(job_url, stream=True, timeout=request_timeout)
        ct = rr.headers.get("Content-Type", "")
        cd = rr.headers.get("Content-Disposition", "")

        # If we got a file, save it and return
        if "attachment" in cd.lower():
            suggested = None
            if "filename=" in cd:
                suggested = cd.split("filename=")[-1].strip('"; ')
            final_name = suggested or filename
            final_path = os.path.join(dest_dir, final_name)
            tmp = final_path + ".part"
            with open(tmp, "wb") as f:
                for chunk in rr.iter_content(chunk_size=1 << 15):
                    if chunk:
                        f.write(chunk)
            os.replace(tmp, final_path)
            print(f"Saved ProtVar download → {final_path}")
            return final_path

        # Otherwise, try to read JSON status and keep polling unless error
        try:
            status_json = rr.json()
        except Exception:
            status_json = None

        if isinstance(status_json, dict):
            status = status_json.get("status")
            if status != last_status:
                print(f"  job status: {status}")
                last_status = status
            # Heuristic: treat negative status as error
            if isinstance(status, int) and status < 0:
                raise RuntimeError(f"ProtVar reported job error (status={status}).")
        else:
            # Unexpected non-JSON, non-file response: short backoff and retry
            time.sleep(poll_every)
            continue

        time.sleep(poll_every)

    raise TimeoutError(f"Timed out waiting for ProtVar download job after {poll_timeout}s: {job_url}")

In [6]:
ACC = "P05067"          # UniProt accession
DL_PATH = protvar_bulk_download_for_accession(
    ACC,
    include_function=True,
    include_population=True,
    include_structure=False,
    assembly="AUTO",
    dest_dir="protvar_downloads",
    force=False,
    poll_timeout=1800,
)
print(DL_PATH)

Requesting ProtVar bulk download job…
  POST https://www.ebi.ac.uk/ProtVar/api/download  params={'inputType': 'PROTEIN_ACCESSION', 'function': 'true', 'population': 'true', 'structure': 'false', 'assembly': 'AUTO', 'jobName': 'P05067-fun-pop-1761684597'}  body='P05067'


HTTPError: 500 Server Error: Internal Server Error for url: https://www.ebi.ac.uk/ProtVar/api/download?inputType=PROTEIN_ACCESSION&function=true&population=true&structure=false&assembly=AUTO&jobName=P05067-fun-pop-1761684597

In [7]:
# ==== Fetch + transform ====
ACC = "O15552"          # UniProt accession
PAGE_SIZE = 1000        # 10..1000
CACHE_PATH = f"{ACC}_protvar.json"

client = ProtVarClient()
bundle = client.fetch_all(ACC, page_size=PAGE_SIZE, cache_path=CACHE_PATH, force=False, verbose=True)

# Print a concise, non-redundant protein summary once
summary = protvar_protein_summary(bundle, accession=ACC, canonical_only=True)
print_protein_summary(summary)

# Variant-level table (lean)
df = protvar_to_dataframe(bundle, accession=ACC, canonical_only=True)
print(f"Flattened {len(df):,} variant rows.")

# Fetch mmCIF file from AFDB
MODEL_VERSION = "v6"   # keep aligned with your current AFDB release
MODEL_DIR = "models"   # change if you prefer another folder

cif_path = download_af_model(ACC, dest_dir=MODEL_DIR, version=MODEL_VERSION, fragment=1, ext="cif", force=False)
print(f"Model path: {cif_path}")

Using cached ProtVar data: O15552_protvar.json
Protein accession: O15552 | Protein name: Free fatty acid receptor 2 | Gene: FFAR2 | Ensembl gene: ENSG00000126262 | Chromosome: 19
Flattened 2,970 variant rows.
Using cached AlphaFold model: models/AF-O15552-F1-model_v6.cif
Model path: models/AF-O15552-F1-model_v6.cif


In [8]:
df

,Position,Reference AA,Variant AA,Consequence,Codon change,Amino acid change,Genomic position,Reference allele,Alternate allele,CADD,Allele frequency,Conservation score,AM pathogenicity,AM class,ESM score,ENSP,ENST
0,1,Met,Arg,missense,aUg/aGg,Met/Arg,35449716,T,G,11.580,NaN,0.863,0.1551,BENIGN,-3.455,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
1,1,Met,Ile,missense,auG/auA,Met/Ile,35449717,G,A,2.369,NaN,0.863,0.1311,BENIGN,-3.258,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
2,1,Met,Ile,missense,auG/auC,Met/Ile,35449717,G,C,1.359,NaN,0.863,0.1311,BENIGN,-3.258,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
3,1,Met,Ile,missense,auG/auU,Met/Ile,35449717,G,T,1.432,NaN,0.863,0.1311,BENIGN,-3.258,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
4,1,Met,Leu,missense,Aug/Cug,Met/Leu,35449715,A,C,14.330,NaN,0.863,0.0638,BENIGN,-2.144,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2965,330,Glu,Glu,synonymous,gaG/gaA,Glu/Glu,35450704,G,A,0.434,NaN,0.814,NaN,None,NaN,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
2966,330,Glu,Gly,missense,gAg/gGg,Glu/Gly,35450703,A,G,19.520,NaN,0.814,0.0827,BENIGN,-1.340,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
2967,330,Glu,Lys,missense,Gag/Aag,Glu/Lys,35450702,G,A,20.300,NaN,0.814,0.1116,BENIGN,-2.485,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3
2968,330,Glu,Ter,stop gained,Gag/Uag,Glu/Ter,35450702,G,T,37.000,NaN,0.814,NaN,None,NaN,ENSP00000246549.2; ENSP00000473159.1,ENST00000246549.2; ENST00000599180.3


In [9]:
# ==== Discrete pLDDT bands (ChimeraX-style) — robust by residue lists ====
import re
import py3Dmol

# Load the model text we already downloaded
with open(cif_path, "r", encoding="utf-8") as f:
    model_text = f.read()

fmt = "pdb" if model_text.lstrip().startswith(("ATOM", "HETATM")) else "cif"

# Colours (your scheme)
COL_ORANGE   = "#FF7D45"  # < 50
COL_YELLOW   = "#FFDB13"  # 50–70
COL_LBLUE    = "#66CBF3"  # 70–90
COL_DBLUE    = "#0054D7"  # ≥ 90

def extract_residue_plddt(text: str) -> dict[int, float]:
    """
    Extract per-residue pLDDT (stored in B-factor). AF mmCIF has a pattern:
    '... 1.00 <B> ? <resno> ...'
    We collect B by residue number; if multiple atoms appear, we take the mean.
    """
    b_by_res = {}
    n_by_res = {}
    for m in re.finditer(r"\s1\.00\s+([0-9]+(?:\.[0-9]+)?)\s+\?\s+(\d+)\s", text):
        b = float(m.group(1))
        resi = int(m.group(2))
        b_by_res[resi] = b_by_res.get(resi, 0.0) + b
        n_by_res[resi] = n_by_res.get(resi, 0) + 1
    # mean per residue (usually identical across atoms in AF models)
    return {resi: b_by_res[resi] / n_by_res[resi] for resi in b_by_res}

def split_into_bands(res_plddt: dict[int, float]):
    orange, yellow, lblue, dblue = [], [], [], []
    for resi, b in res_plddt.items():
        if b < 50:
            orange.append(resi)
        elif b < 70:
            yellow.append(resi)
        elif b < 90:
            lblue.append(resi)
        else:
            dblue.append(resi)
    return orange, yellow, lblue, dblue

def _chunk(lst, n=500):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# Compute bands once
res_plddt = extract_residue_plddt(model_text)
orange, yellow, lblue, dblue = split_into_bands(res_plddt)
print(f"Residues per band → <50: {len(orange)}, 50–70: {len(yellow)}, 70–90: {len(lblue)}, ≥90: {len(dblue)}")

# Build viewer and colour per band
v = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
v.addModel(model_text, fmt)

# Base style so unassigned bits are visible
v.setStyle({}, {"cartoon": {"color": "lightgrey"}})

# Overlay per band using residue lists (robust)
for lst, col in [(orange, COL_ORANGE), (yellow, COL_YELLOW), (lblue, COL_LBLUE), (dblue, COL_DBLUE)]:
    for chunk in _chunk(sorted(lst), 1000):
        v.addStyle({"resi": chunk}, {"cartoon": {"color": col}})

v.zoomTo()
v.show()

Residues per band → <50: 24, 50–70: 21, 70–90: 52, ≥90: 233


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# ==== One viewer: pLDDT + filtered variant pins (missense/synonymous) ====
import re
import numpy as np
import pandas as pd
import py3Dmol
from IPython.display import display

# --- Load model text and decide parser format (reuse cif_path) ---
with open(cif_path, "r", encoding="utf-8") as f:
    model_text = f.read()
fmt = "pdb" if model_text.lstrip().startswith(("ATOM", "HETATM")) else "cif"

# --- Classify consequence into variant types we care about ---
def _classify_variant_type(consequence) -> str:
    """
    Map ProtVar 'Consequence' to a coarse type: 'missense', 'synonymous', 'other'.
    Accepts strings or lists; case-insensitive.
    """
    if isinstance(consequence, list):
        text = " ".join([str(x) for x in consequence if x is not None])
    else:
        text = "" if consequence is None else str(consequence)
    s = text.lower()
    if "synonymous" in s:
        return "synonymous"
    if "missense" in s or "nonsynonymous" in s:
        return "missense"
    return "other"

# Derive a tidy working table (copy to avoid mutating df)
_variants = df.copy()
_variants = _variants.dropna(subset=["Position"]).copy()
_variants["Position"] = _variants["Position"].astype(int)
_variants["Variant type"] = _variants["Consequence"].map(_classify_variant_type)

# Optional: a useful human label like A123T, if both residues present
def _mk_label(row):
    r = (row.get("Reference AA") or "").strip()
    v = (row.get("Variant AA") or "").strip()
    p = int(row["Position"])
    return f"{r}{p}{v}" if r and v else str(p)

_variants["Label"] = _variants.apply(_mk_label, axis=1)

def _subset_positions(variant_type: str, max_positions: int = 200, random_sample: bool = True):
    """
    Return (positions, label_map, table) for the chosen filter.
    - positions: list[int] unique residue indices to highlight
    - label_map: dict[pos] -> label string
    - table: small DataFrame describing highlighted variant rows
    """
    if variant_type.lower() == "missense":
        sel = _variants[_variants["Variant type"] == "missense"]
    elif variant_type.lower() == "synonymous":
        sel = _variants[_variants["Variant type"] == "synonymous"]
    else:
        sel = _variants  # all

    # Collapse to unique positions, keeping first representative for the label/table
    rep = (
        sel.sort_values(["Position"])
           .drop_duplicates(subset=["Position"], keep="first")
           .copy()
    )
    if random_sample and len(rep) > max_positions:
        rep = rep.sample(n=max_positions, random_state=42).sort_values("Position")
    else:
        rep = rep.head(max_positions)

    positions = rep["Position"].astype(int).tolist()
    label_map = dict(zip(rep["Position"].astype(int), rep["Label"]))
    table = rep[["Position", "Reference AA", "Variant AA", "Consequence", "CADD", "Allele frequency", "ENSP", "ENST"]]
    return positions, label_map, table

def show_structure_with_variants(
    model_text: str,
    fmt: str,
    variant_type: str = "missense",   # 'missense' | 'synonymous' | 'all'
    max_positions: int = 200,
    random_sample: bool = True,
    sphere_radius: float = 1.0,
    sphere_color: str = "red",
):
    """
    Render a single viewer: pLDDT gradient + red pins at filtered variant positions.
    Also displays a compact table of highlighted variants.
    """
    positions, label_map, table = _subset_positions(variant_type, max_positions, random_sample)

    v = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
    v.addModel(model_text, fmt)

    # pLDDT gradient (reliable)
    v.setStyle({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})

    # Overlay spheres at CA atoms for selected positions
    for pos in positions:
        v.addStyle({"resi": int(pos), "atom": "CA"},
                   {"sphere": {"radius": float(sphere_radius), "color": sphere_color}})

    v.zoomTo()
    v.show()

    # Show the little table under the viewer
    display(table.reset_index(drop=True))
    print(f"Highlighted {len(positions)} position(s) (filter: {variant_type}).")

# --- Try to provide a tiny interactive control if ipywidgets is available ---
try:
    import ipywidgets as W
    def _on_change(variant_type, max_positions, random_sample):
        show_structure_with_variants(
            model_text=model_text, fmt=fmt,
            variant_type=variant_type,
            max_positions=max_positions,
            random_sample=random_sample,
            sphere_radius=1.0, sphere_color="red",
        )
    ui = W.VBox([
        W.HBox([
            W.Dropdown(options=["missense", "synonymous", "all"], value="missense", description="Variant type:"),
            W.IntSlider(value=50, min=10, max=500, step=10, description="Max pins:", readout=True),
            W.Checkbox(value=True, description="Random sample"),
        ])
    ])
    out = W.interactive_output(_on_change, {
        "variant_type": ui.children[0].children[0],
        "max_positions": ui.children[0].children[1],
        "random_sample": ui.children[0].children[2],
    })
    display(ui, out)
except Exception:
    # Fallback: render once with defaults (missense, 50 pins)
    show_structure_with_variants(
        model_text=model_text, fmt=fmt,
        variant_type="missense", max_positions=50, random_sample=True,
        sphere_radius=1.0, sphere_color="red",
    )

Output()